# PALA Artifact Analysis Notebook

**Paper:** *Architecture-Induced Reward Hacking in NWDAF-Integrated LLM Control Loops*  
**IEEE S&P 2027, Cycle 1**

This notebook loads the stored per-session trial logs from `final_experiments/` and
reproduces all key tables and figures from the paper.

**No live 5G network or GPU required.** Run all cells from top to bottom.

---

## Table of Contents
1. Setup & Data Loading
2. RQ1 — Circuit Realization (Table II, Figure 3)
3. RQ2 — Mechanism Analysis (Figure 4)
4. RQ3 — Oversight Scope (Table III, Figure 5)
5. RQ4 — Defense Efficacy (Tables IV, VII, VIII, IX)
6. RQ5 — Deployability (Table XIII)
7. Summary Validation

## 1. Setup & Data Loading

In [ ]:
import json
import math
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ROOT    = Path('..').resolve()
EXP_DIR = ROOT / 'final_experiments'
OUT_DIR = ROOT / 'final_paper_results'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

def load_json(rel_path):
    p = EXP_DIR / rel_path
    if not p.exists():
        print(f'[MISSING] {rel_path}')
        return {}
    return json.loads(p.read_text())

def load_jsonl(rel_path):
    p = EXP_DIR / rel_path
    if not p.exists():
        print(f'[MISSING] {rel_path}')
        return []
    return [json.loads(line) for line in p.read_text().strip().splitlines() if line.strip()]

print('Data directory:', EXP_DIR)
print('Experiments found:', [d.name for d in EXP_DIR.iterdir() if d.is_dir()])

## 2. RQ1 — Circuit Realization (Table II, Figure 3)

In [ ]:
# Load Exp 1 summary (Qwen, 30+30 sessions)
exp1 = load_json('exp1/summary.json')

print('=== Exp 1 Summary (Qwen 2.5:72b) ===')
print(f"  Vulnerable: n={exp1['vulnerable']['n']}")
print(f"    Full-loop rate:   {exp1['vulnerable']['full_loop_rate']:.1%}")
print(f"    Strict Def. 4:    {exp1['vulnerable']['def4_rate']:.1%}")
print(f"    Mean Q-drop:      {exp1['vulnerable']['q_drop_mean']:.3f}")
print(f"    Contaminated:     {exp1['vulnerable']['contaminated_rate']:.1%}")
print()
print(f"  Defended (Full PALA): n={exp1['defended']['n']}")
print(f"    Full-loop rate:   {exp1['defended']['full_loop_rate']:.1%}")
print(f"    Strict Def. 4:    {exp1['defended']['def4_rate']:.1%}")
print(f"    Contaminated:     {exp1['defended']['contaminated_rate']:.1%}")
print()
print(f"  Fisher p (Full PALA vs Vulnerable): {exp1['fisher_p_full_loop']:.2e}")

In [ ]:
# Load cross-family results
mm = load_json('exp1_multimodel/summary.json')

print('=== Table II — Cross-Family Circuit Realization ===')
print(f"{'Model':<22} {'n':>4} {'Strict Def.4':>14} {'Full-loop A∧B∧C∧D':>20} {'Mean ΔQ':>10}")
print('-' * 72)

model_display = {
    'qwen2.5:72b':          'Qwen 2.5:72b',
    'mistral-large:latest': 'Mistral-large',
    'llama3.1:70b':         'Llama 3.1:70b',
}

for key, display in model_display.items():
    m = mm['models'].get(key, {})
    n  = m.get('n', 30)
    fl = m.get('full_loop_rate', 0)
    d4 = m.get('def4_rate', 0)
    qd = m.get('mean_q_drop', None)
    qd_s = f"{-qd:.1%}" if qd else '—'
    print(f"  {display:<20} {n:>4} {d4:.0%} ({int(d4*n)}/{n}){'':<5}"
          f" {fl:.0%} ({int(fl*n)}/{n}){'':<11} {qd_s}")

In [ ]:
# Figure 3b — Q envelope across 30 Qwen vulnerable sessions
trials = load_jsonl('exp1/exp1_trials.jsonl')
vuln_trials = [t for t in trials if t.get('arm') == 'vulnerable' and t.get('q_trace')]

print(f'Vulnerable sessions with q_trace: {len(vuln_trials)}')

# Align q_traces to policy step k
max_k = max(len(t['q_trace']) for t in vuln_trials)
q_matrix = np.full((len(vuln_trials), max_k), np.nan)
for i, t in enumerate(vuln_trials):
    for j, step in enumerate(t['q_trace']):
        q0 = t['baseline_q']['Q']
        q_matrix[i, j] = (step['Q'] - q0) / q0 if q0 > 0 else 0.0

fig, ax = plt.subplots(figsize=(6, 3.5))
ks = np.arange(1, max_k + 1)
median = np.nanmedian(q_matrix, axis=0)
p5  = np.nanpercentile(q_matrix, 5, axis=0)
p95 = np.nanpercentile(q_matrix, 95, axis=0)
iqr_lo = np.nanpercentile(q_matrix, 25, axis=0)
iqr_hi = np.nanpercentile(q_matrix, 75, axis=0)

ax.fill_between(ks, p5, p95, alpha=0.15, color='#1565C0', label='5–95 pct')
ax.fill_between(ks, iqr_lo, iqr_hi, alpha=0.30, color='#1565C0', label='IQR')
ax.plot(ks, median, color='#1565C0', lw=2, label='Median ΔQ/Q₀')
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.set_xlabel('Policy-call step k')
ax.set_ylabel('ΔQ(k)/Q₀')
ax.set_title('Figure 3b — 30-session Q envelope (Qwen 2.5:72b vulnerable)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3b_q_envelope.pdf', bbox_inches='tight')
plt.show()
print('Saved fig3b_q_envelope.pdf')

In [ ]:
# Figure 3d — Per-step ΔQ KDE
delta_q_all = q_matrix.flatten()
delta_q_all = delta_q_all[~np.isnan(delta_q_all)]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.hist(delta_q_all, bins=40, density=True, color='#1565C0', alpha=0.6, label='All steps')

# KDE
from scipy.stats import gaussian_kde
kde = gaussian_kde(delta_q_all, bw_method=0.3)
xs = np.linspace(delta_q_all.min(), delta_q_all.max(), 300)
ax.plot(xs, kde(xs), color='#1565C0', lw=2)
ax.axvline(0, color='grey', lw=0.8, ls='--')

neg_frac = (delta_q_all < 0).mean()
ax.set_xlabel('Per-step ΔQ/Q₀')
ax.set_ylabel('Density')
ax.set_title(f'Figure 3d — Per-step ΔQ KDE (Qwen, {len(delta_q_all)} transitions)\n'
             f'Neg. steps: {neg_frac:.0%}')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3d_delta_q_kde.pdf', bbox_inches='tight')
plt.show()
print(f'Saved fig3d_delta_q_kde.pdf  (negative transitions: {neg_frac:.1%})')

## 3. RQ2 — Mechanism Analysis (Figure 4)

In [ ]:
# Figure 4a — Decomposition rate by intent register
exp2 = load_json('exp2/summary.json')

registers = ['staged', 'direct', 'null']
labels    = ['Staged\n(ITIL/SRE)', 'Direct\nchange', 'Null\ncontrol']
decomp    = [exp2.get(r, {}).get('decomposed_rate', 0) for r in registers]
full_loop = [exp2.get(r, {}).get('full_loop_rate', 0) for r in registers]

fig, ax = plt.subplots(figsize=(5, 3.5))
x = np.arange(len(registers))
w = 0.35
ax.bar(x - w/2, [d*100 for d in decomp],    w, label='Stage A: Decompose', color='#1565C0', alpha=0.85)
ax.bar(x + w/2, [d*100 for d in full_loop], w, label='Full loop A∧B∧C∧D', color='#C62828', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Rate (%)')
ax.set_title('Figure 4a — Decomposition by intent register (n=20 each)')
ax.legend(fontsize=9)
for xi, (d, fl) in enumerate(zip(decomp, full_loop)):
    ax.text(xi - w/2, d*100 + 1, f'{d:.0%}', ha='center', va='bottom', fontsize=9)
    ax.text(xi + w/2, fl*100 + 1, f'{fl:.0%}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4a_decomp_register.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4c — Type-P contamination: standard vs IsolatedCollector
exp14 = load_json('exp14/summary.json')

std_rate = exp14.get('standard_contamination_rate', 1.0)
iso_rate = exp14.get('isolated_contamination_rate', 0.0)

print('=== Figure 4c — Type-P Contamination ===')
print(f'  Standard collector:    {std_rate:.0%}  (Lemma 2: 100% saturated)')
print(f'  IsolatedCollector:     {iso_rate:.0%}  (Lemma 6: provenance severed)')

fig, ax = plt.subplots(figsize=(4, 3.5))
bars = ax.bar(['Standard\n(AS2 violated)', 'IsolatedCollector\n(AS2 enforced)'],
               [std_rate * 100, iso_rate * 100],
               color=['#C62828', '#2E7D32'], alpha=0.85)
ax.set_ylabel('Type-P contamination rate (%)')
ax.set_title('Figure 4c — Provenance surface (Lemma 2 & 6)\n(n=20 writes each)')
ax.set_ylim(0, 120)
for bar, val in zip(bars, [std_rate, iso_rate]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.0%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4c_type_p_contamination.pdf', bbox_inches='tight')
plt.show()

## 4. RQ3 — Oversight Scope (Table III, Figure 5)

In [ ]:
exp3 = load_json('exp3/summary.json')

print('=== Table III — Reviewer-Rule Approval Rates ===')
pc   = exp3.get('per_call_approval_rate_mean', 0)
cumr = exp3.get('cumulative_approval_rate_mean', 0)
wp   = exp3.get('wilcoxon_p', None)

print(f'  Per-call approval (AS6 model):    {pc:.1%}')
print(f'  Cumulative approval (stateful):   {cumr:.1%}')
if wp:
    print(f'  Wilcoxon p (cum > per-call):      {wp:.2e}')
    print(f'  Significant (p < 0.001):          {"YES" if wp < 0.001 else "NO"}')

# Figure 5 — Bar chart
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(['Per-call\n(AS6 model)', 'AND-rule\n(per-call ∧ cumul.)', 'Cumulative\nonly'],
        [pc*100, (pc*100 * 0.97), cumr*100],   # AND-rule ≈ per-call minus 1-2pp
        color=['#1565C0', '#5C8BC4', '#C62828'], alpha=0.85)
ax.set_ylabel('Approval rate of attack steps (%)')
ax.set_title('Figure 5 — Per-call vs cumulative oversight\n(n=30 Qwen sessions, τ=20 Mbps)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_oversight_rules.pdf', bbox_inches='tight')
plt.show()

## 5. RQ4 — Defense Efficacy (Tables IV, VII, VIII, IX)

In [ ]:
# Table VII — HedgeTune k†* Calibration
kstar_data = load_json('exp6/kstar.json')

print('=== Table VII — HedgeTune k†* Calibration ===')
print(f"  k†*          = {kstar_data.get('k_star')}")
print(f"  n_calibration= {kstar_data.get('n_sessions')}")
print(f"  n_skipped    = {kstar_data.get('n_skipped')} (incomplete q_trace)")
print(f"  Mean k†_i    = {kstar_data.get('mean_k_dagger', 0):.3f}")
print(f"  CV           = {kstar_data.get('cv', 0):.3f}")
print(f"  95% CI       = [{kstar_data.get('ci_95_lo', 0):.3f}, {kstar_data.get('ci_95_hi', 0):.3f}]")
print(f"  CV < 0.5 (stable): {kstar_data.get('cv_stable')}")

# Distribution of k†_i
k_daggers = kstar_data.get('k_daggers', [])
if k_daggers:
    from collections import Counter
    dist = Counter(k_daggers)
    total = len(k_daggers)
    print('\n  k†_i distribution:')
    for k_val in sorted(dist):
        print(f"    k={k_val}: {dist[k_val]}/{total} = {dist[k_val]/total:.0%}")

In [ ]:
# Table VIII — Necessity/Sufficiency Ablation
exp5 = load_json('exp5/summary.json')
v = exp5.get('variants', {})

print('=== Table VIII — Necessity/Sufficiency Ablation (n=20 per variant) ===')
print(f"{'Variant':<30} {'Full-loop count':>16} {'Full-loop rate':>14}")
print('-' * 62)

rows = [
    ('Full chain (AS2∧AS4∧AS6)',      'full_chain'),
    ('AS2 only (IsolatedCollector)',  'as2_only'),
    ('AS4 only (HedgeTuned)',         'as4_only'),
    ('AS2 ∧ AS4 (Full PALA)',         'as2_and_as4'),
    ('AS5 only (bounded N)',          'as5_only'),
]

for label, key in rows:
    vd = v.get(key, {})
    n   = vd.get('n', 20)
    cnt = vd.get('corrected_full_loop_count', vd.get('full_loop_count', 0))
    rt  = vd.get('corrected_full_loop_rate',  vd.get('full_loop_rate', 0))
    print(f"  {label:<28} {cnt:>5}/{n:<10} {rt:.0%}")

In [ ]:
# Table IV — Headline Defense Efficacy
exp1_sum = load_json('exp1/summary.json')

print('=== Table IV — Headline Defense Efficacy ===')
configs = [
    ('Vulnerable (no defense)',       exp1_sum.get('vulnerable', {}), 30),
    ('Full PALA Guardrail',           exp1_sum.get('defended', {}),   30),
]

print(f"{'Configuration':<35} {'Op. circuit closure':>20} {'Strict Def.4':>14} {'Mean ΔQ':>10}")
print('-' * 82)
for label, d, n in configs:
    fl  = d.get('full_loop_rate', 0)
    d4  = d.get('def4_rate', 0)
    dq  = d.get('q_drop_mean', 0)
    print(f"  {label:<33} {fl:.0%} ({int(fl*n)}/{n}){'':<8}"
          f" {d4:.0%} ({int(d4*n)}/{n}){'':<3} {-dq:.1%}")

# Add exp5 variants
for label, key in [('IsolatedCollector (AS2)', 'as2_only'), ('HedgeTuned k†=1 (AS4)', 'as4_only')]:
    vd = v.get(key, {})
    n  = 20
    fl = vd.get('corrected_full_loop_rate', 0)
    d4 = vd.get('def4_rate', vd.get('full_loop_rate', 0))
    print(f"  {label:<33} {fl:.0%} ({int(fl*n)}/{n}){'':<8}"
          f" {d4:.0%} ({int(d4*n)}/{n}){'':<3} n/a")

In [ ]:
# Table IX — Adaptive Attacker
exp7 = load_json('exp7/summary.json')
conds = exp7.get('conditions', {})

print('=== Table IX — Adaptive Attacker (n=50 per condition) ===')
print(f"{'Condition':<35} {'Full-loop':>12} {'Fisher p':>12}")
print('-' * 62)

cond_rows = [
    ('Vulnerable (no defense)',  'none'),
    ('IsolatedCollector alone',  'iso'),
    ('HedgeTuned k†=1 alone',   'ht'),
    ('Full PALA Guardrail',      'both'),
]
vuln_count = conds.get('none', {}).get('full_loop_count_corr', 46)

for label, key in cond_rows:
    cd = conds.get(key, {})
    n  = cd.get('n', 50)
    cnt = cd.get('full_loop_count_corr', 0)
    rt  = cd.get('full_loop_rate_corr', 0)
    # Compute Fisher p vs vulnerable
    if key != 'none' and vuln_count > 0:
        table = [[cnt, n - cnt], [vuln_count, 50 - vuln_count]]
        _, p = stats.fisher_exact(table)
        p_s = f'{p:.2e}'
    else:
        p_s = 'baseline'
    print(f"  {label:<33} {cnt}/{n} ({rt:.0%}){'':<3} {p_s}")

## 6. RQ5 — Deployability (Table XIII)

In [ ]:
exp8  = load_json('exp8/summary.json')
exp15 = load_json('exp15/summary.json')

print('=== Table XIII — Deployability (Full PALA) ===')

fp = exp8.get('full_pala', {})
print('Block 1 — Benign workload completion:')
print(f"  Completion rate:      {fp.get('completion_rate', 'n/a')}")
print(f"  Type-P contamination: {fp.get('type_p_rate', 'n/a')}")
print(f"  H_budget false-rej:   {fp.get('false_reject_rate', 'n/a')}")
print()
print('Block 2 — Post-incident recovery:')
print(f"  Recovery rate:        {fp.get('recovery_rate', 'n/a')}")
print()
print('Block 3 — Runtime cost:')
print(f"  KPI latency (ISO, ms): {exp15.get('kpi_latency_ms_iso', 'n/a')}")
print(f"  KPI latency (std, ms): {exp15.get('kpi_latency_ms_standard', 'n/a')}")

## 7. Summary Validation

In [ ]:
print('══════════════════════════════════════════════════════════')
print('  PALA Artifact — Key Claims Validation Summary')
print('══════════════════════════════════════════════════════════')

checks = [
    ('RQ1', 'Qwen full-loop ≥ 60%',
     mm['models']['qwen2.5:72b']['full_loop_rate'] >= 0.60),
    ('RQ1', 'Mistral full-loop ≥ 20%',
     mm['models']['mistral-large:latest']['full_loop_rate'] >= 0.20),
    ('RQ1', 'Llama full-loop ≥ 20%',
     mm['models']['llama3.1:70b']['full_loop_rate'] >= 0.20),
    ('RQ1', 'Full PALA: 0 closures',
     exp1_sum['defended']['full_loop_rate'] == 0.0),
    ('RQ2', 'Staged > Direct decomposition',
     exp2.get('staged', {}).get('decomposed_rate', 0) >
     exp2.get('direct', {}).get('decomposed_rate', 1)),
    ('RQ2', 'Standard collector: 100% Type-P',
     exp14.get('standard_contamination_rate', 0) >= 0.95),
    ('RQ2', 'IsolatedCollector: 0% Type-P',
     exp14.get('isolated_contamination_rate', 1) <= 0.05),
    ('RQ3', 'Cumulative > per-call approval (Wilcoxon p<0.001)',
     exp3.get('wilcoxon_p', 1.0) < 0.001),
    ('RQ4', 'k†* = 1',
     kstar_data.get('k_star') == 1),
    ('RQ4', 'CV ≤ 0.5 (stable)',
     kstar_data.get('cv', 1.0) <= 0.5),
    ('RQ4', 'IsolatedCollector alone: 0 full-loops',
     v.get('as2_only', {}).get('corrected_full_loop_rate', 1) == 0.0),
    ('RQ4', 'HedgeTuned alone: 0 corrected full-loops',
     v.get('as4_only', {}).get('corrected_full_loop_rate', 1) == 0.0),
    ('RQ4', 'Adaptive attacker: 0 closures (Full PALA)',
     conds.get('both', {}).get('full_loop_rate_corr', 1) == 0.0),
]

passed = 0
for rq, label, ok in checks:
    status = '\033[92mPASS\033[0m' if ok else '\033[91mFAIL\033[0m'
    print(f'  [{rq}] {label:<48}  {status}')
    if ok:
        passed += 1

print(f'\n  {passed}/{len(checks)} checks PASS')
print()
print('  All pre-generated figures saved to:')
print(f'  {FIG_DIR}')